# Analyse de l'ontologie Gene Ontology (GO)

Comparaison des versions **oct. 2025** et **jan. 2026** sur le domaine *DNA repair* (`GO:0006281`).

In [1]:
import sys
from pathlib import Path

import pandas as pd

# Les scripts utilitaires sont dans le même dossier que ce notebook
ANALYSE_DIR = Path().resolve()
sys.path.insert(0, str(ANALYSE_DIR))

from load_ontologies import (
    GO_NS,
    PATH_GO_NEW,
    PATH_GO_OLD,
    get_stats,
    load_ontology,
)

## Section 1 — Exploration générale des deux versions GO

Chargement des deux fichiers OWL et calcul des statistiques globales via `get_stats()`.

> **nb_axiomes** : axiomes logiques (SubClassOf + EquivalentClasses + DisjointWith), aligné avec Protégé.

In [2]:
print("Chargement GO oct. 2025 …")
onto_old = load_ontology(PATH_GO_OLD)

print("Chargement GO jan. 2026 …")
onto_new = load_ontology(PATH_GO_NEW)

stats_old = get_stats(onto_old, "GO_10-25", PATH_GO_OLD)
stats_new = get_stats(onto_new, "GO_01-26", PATH_GO_NEW)

rows = {
    "nb_classes":    [stats_old["nb_classes"],              stats_new["nb_classes"]],
    "nb_proprietes": [stats_old["proprietes"]["total"],     stats_new["proprietes"]["total"]],
    "  obj_props":   [stats_old["proprietes"]["object_properties"],
                      stats_new["proprietes"]["object_properties"]],
    "  data_props":  [stats_old["proprietes"]["data_properties"],
                      stats_new["proprietes"]["data_properties"]],
    "  annot_props": [stats_old["proprietes"]["annotation_properties"],
                      stats_new["proprietes"]["annotation_properties"]],
    "nb_axiomes":    [stats_old["nb_axiomes"],               stats_new["nb_axiomes"]],
    "nb_individus":  [stats_old["nb_individus"],             stats_new["nb_individus"]],
}

df_stats = pd.DataFrame(rows, index=["GO_10-25", "GO_01-26"])
df_stats.index.name = "version"

delta = df_stats.loc["GO_01-26"] - df_stats.loc["GO_10-25"]
df_stats.loc["Δ (01-26 − 10-25)"] = delta

df_stats

Chargement GO oct. 2025 …
Chargement GO jan. 2026 …


,nb_classes,nb_proprietes,obj_props,data_props,annot_props,nb_axiomes,nb_individus
version,,,,,,,
GO_10-25,51842,71,9,0,62,86960,0
GO_01-26,51897,75,9,0,66,84682,0
Δ (01-26 − 10-25),55,4,0,0,4,-2278,0


## Section 2 — Analyse quantitative du domaine DNA repair (`GO:0006281`)

Pour chaque version on calcule :
- **nb_classes_domaine** : nombre de termes dans le sous-arbre `GO:0006281` (hiérarchie **is_a** uniquement)
- **nb_classes_nouvelles** : termes apparus dans la nouvelle version
- **nb_classes_supprimées** : termes absents de la nouvelle version
- **nb_deprecated** : termes marqués `owl:deprecated true`
- **nb_hierarchie_changée** : termes dont les parents `is_a` directs ont changé

> La traversée utilise SPARQL 1.1 (`rdfs:subClassOf*`) sur le world owlready2.
> On compte uniquement la hiérarchie **is_a** (rdfs:subClassOf), pas part_of/regulates.

In [3]:
DOMAIN_ROOT_ID = "GO:0006281"


def go_id_to_iri(go_id: str) -> str:
    return GO_NS + go_id.replace(":", "_")


def get_descendants(onto, root_go_id: str) -> set:
    """SPARQL 1.1 rdfs:subClassOf* sur le world owlready2 — retourne un ensemble d'IRI (str)."""
    root_iri = go_id_to_iri(root_go_id)
    try:
        q = "SELECT DISTINCT ?sub WHERE { ?sub rdfs:subClassOf* <%s> . FILTER(ISIRI(?sub)) }" % root_iri
        rows = list(onto.world.sparql(q))
        return {str(row[0]) for row in rows}
    except Exception as e:
        print(f"  [WARN] SPARQL descendants échoué pour {root_go_id}: {e}")
        return set()


def count_deprecated(onto, class_iris: set) -> int:
    return sum(
        1 for cls in onto.classes()
        if cls.iri in class_iris and getattr(cls, "deprecated", None)
    )


def get_direct_parents(cls) -> set:
    return {p.iri for p in cls.is_a if getattr(p, "iri", None)}


def count_hierarchy_changes(onto_old, onto_new, iris_common: set) -> int:
    old_idx = {cls.iri: cls for cls in onto_old.classes() if cls.iri in iris_common}
    new_idx = {cls.iri: cls for cls in onto_new.classes() if cls.iri in iris_common}
    return sum(
        1 for iri in iris_common
        if old_idx.get(iri) and new_idx.get(iri)
        and get_direct_parents(old_idx[iri]) != get_direct_parents(new_idx[iri])
    )

In [4]:
print(f"Extraction des descendants de {DOMAIN_ROOT_ID} …")
iris_old = get_descendants(onto_old, DOMAIN_ROOT_ID)
iris_new = get_descendants(onto_new, DOMAIN_ROOT_ID)

iris_added   = iris_new - iris_old
iris_removed = iris_old - iris_new
iris_common  = iris_old & iris_new

print("Comptage des classes dépréciées …")
n_depr_old = count_deprecated(onto_old, iris_old)
n_depr_new = count_deprecated(onto_new, iris_new)

print("Détection des changements de hiérarchie …")
n_hier_changed = count_hierarchy_changes(onto_old, onto_new, iris_common)

results = {
    "nb_classes_domaine":    [len(iris_old),    len(iris_new)],
    "nb_classes_nouvelles":  ["—",              len(iris_added)],
    "nb_classes_supprimées": [len(iris_removed), "—"],
    "nb_deprecated":         [n_depr_old,        n_depr_new],
    "nb_hierarchie_changée": ["—",               n_hier_changed],
}

df_quant = pd.DataFrame(results, index=["GO_10-25", "GO_01-26"])
df_quant.index.name = "version"
df_quant

Extraction des descendants de GO:0006281 …
Comptage des classes dépréciées …
Détection des changements de hiérarchie …


,nb_classes_domaine,nb_classes_nouvelles,nb_classes_supprimées,nb_deprecated,nb_hierarchie_changée
version,,,,,
GO_10-25,44,—,1,0,—
GO_01-26,46,3,—,0,0
